# 20. 전체 인디게임 가격 분포

**분석 목적:** `steam_indie_games`와 `steam_indie_games_silence` 전체 데이터를 합산하여,
현재 Steam 인디게임 시장의 가격대별 공급 구조를 파악한다.

**사용 데이터:**
- `data/preprocessed/steam_indie_games.csv` — 초기 반응 그룹 (리뷰 10개 이상)
- `data/preprocessed/steam_indie_games_silence.csv` — 침묵 그룹 (리뷰 10개 미만)

**분석 관점:** 시장현황 — 인디 개발사가 주로 선택하는 가격대와 시장 내 가격 분포 구조 파악

In [1]:
import warnings

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

PRICE_BINS   = [0, 5, 10, 15, 20, 30, 60, float('inf')]
PRICE_LABELS = ['$1~$5', '$5~$10', '$10~$15', '$15~$20', '$20~$30', '$30~$60', '$60+']
COLOR_BAR    = '#4C72B0'

## 1. 데이터 로드 및 병합

In [2]:
df_response = pd.read_csv('../../data/preprocessed/steam_indie_games.csv')
df_silence  = pd.read_csv('../../data/preprocessed/steam_indie_games_silence.csv')

df_response['group'] = '초기 반응 (≥10개)'
df_silence['group']  = '침묵 (<10개)'

df = pd.concat([df_response, df_silence], ignore_index=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

print(f'초기 반응 그룹 : {len(df_response):,}개')
print(f'침묵 그룹      : {len(df_silence):,}개')
print(f'전체           : {len(df):,}개')
print(f'\n가격 기초 통계')
print(df['price'].describe().rename({
    'count': '게임 수', 'mean': '평균', 'std': '표준편차',
    'min': '최솟값', '25%': '1사분위', '50%': '중앙값',
    '75%': '3사분위', 'max': '최댓값',
}).round(2).to_string())

초기 반응 그룹 : 8,730개
침묵 그룹      : 6,676개
전체           : 15,406개

가격 기초 통계
게임 수    15406.00
평균          7.58
표준편차       11.17
최솟값         0.49
1사분위        2.99
중앙값         4.99
3사분위        9.99
최댓값       500.00


## 2. 가격대 구간 분류

In [3]:
df['price_range'] = pd.cut(
    df['price'],
    bins=PRICE_BINS,
    labels=PRICE_LABELS,
    right=False,
)

price_stats = (
    df.groupby('price_range', observed=True)
    .agg(game_count=('appid', 'nunique'))
    .reset_index()
)
price_stats['ratio'] = price_stats['game_count'] / price_stats['game_count'].sum() * 100

display(price_stats.rename(columns={
    'price_range': '가격대', 'game_count': '게임 수', 'ratio': '비율 (%)'
}).round(1).set_index('가격대'))

,게임 수,비율 (%)
가격대,,
$1~$5,8495,55.1
$5~$10,3897,25.3
$10~$15,1575,10.2
$15~$20,928,6.0
$20~$30,383,2.5
$30~$60,93,0.6
$60+,35,0.2


## 3. 가격대별 게임 수 & 비율 — 이중 바 차트

In [4]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(
        x=price_stats['price_range'].astype(str),
        y=price_stats['game_count'],
        marker_color=COLOR_BAR,
        text=price_stats['game_count'].apply(lambda v: f'{v:,}'),
        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(color='white', size=11),
        hovertemplate='<b>%{x}</b><br>게임 수: %{y:,}<extra></extra>',
        name='게임 수',
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=price_stats['price_range'].astype(str),
        y=price_stats['ratio'].round(1),
        mode='lines+markers+text',
        marker=dict(size=8, color='#C44E52'),
        line=dict(color='#C44E52', width=2),
        text=price_stats['ratio'].apply(lambda v: f'{v:.1f}%'),
        textposition='top center',
        textfont=dict(size=10, color='#C44E52'),
        hovertemplate='<b>%{x}</b><br>비율: %{y:.1f}%<extra></extra>',
        name='비율 (%)',
    ),
    secondary_y=True,
)

fig.update_layout(
    title=f'전체 인디게임 가격대 분포<br>'
          f'<sub>전체 {len(df):,}개 게임 / steam_indie_games + steam_indie_games_silence 합산</sub>',
    height=480,
    legend=dict(x=0.75, y=0.95),
)
fig.update_yaxes(title_text='게임 수', secondary_y=False)
fig.update_yaxes(title_text='비율 (%)', secondary_y=True, showgrid=False)
fig.update_xaxes(title_text='가격대')

fig.show()

**해석:** ~$5 구간에 전체 게임의 절반 이상이 집중되어 있어, 인디게임 시장의 가격 경쟁이 저가대에 극단적으로 몰려 있음을 보여준다. 중앙값 가격이 $4.99 수준으로, 대부분의 인디 개발사가 진입 장벽을 낮추는 전략을 선택하고 있다.

## 4. 가격 히스토그램 ($60 이하 유료 게임)

In [5]:
df_paid = df[df['price'] <= 60].copy()

median_price = df_paid['price'].median()
mean_price   = df_paid['price'].mean()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_paid['price'],
    xbins=dict(start=0, end=60, size=1),
    marker_color=COLOR_BAR,
    opacity=0.85,
    hovertemplate='가격: $%{x:.0f}~%{x:.0f}<br>게임 수: %{y:,}<extra></extra>',
    name='게임 수',
))

fig.add_vline(
    x=median_price, line_dash='dash', line_color='#C44E52', line_width=2,
    annotation_text=f'중앙값 ${median_price:.2f}',
    annotation_position='top right',
    annotation_font=dict(size=11, color='#C44E52'),
)
fig.add_vline(
    x=mean_price, line_dash='dot', line_color='#55A868', line_width=2,
    annotation_text=f'평균 ${mean_price:.2f}',
    annotation_position='top left',
    annotation_font=dict(size=11, color='#55A868'),
)

fig.update_layout(
    title='가격 분포 히스토그램 ($60 이하 유료 게임)<br>'
          '<sub>빨간 점선: 중앙값 / 초록 점선: 평균</sub>',
    xaxis_title='가격 (USD)',
    yaxis_title='게임 수',
    bargap=0.05,
    height=420,
    plot_bgcolor='#FAFAFA',
)
fig.update_xaxes(tickprefix='$', dtick=5)

fig.show()

print(f'분석 대상: {len(df_paid):,}개 (전체 {len(df):,}개 중 $60 초과 {len(df) - len(df_paid):,}개 제외)')
print(f'중앙값: ${median_price:.2f}  /  평균: ${mean_price:.2f}')

분석 대상: 15,371개 (전체 15,406개 중 $60 초과 35개 제외)
중앙값: $4.99  /  평균: $7.17


**해석:** 히스토그램은 $0~$5 구간에 급격히 집중되고 이후 급감하는 강한 우편향 분포를 보인다. 평균이 중앙값보다 높은 것은 일부 고가 게임($20 이상)이 평균을 끌어올리기 때문이며, 전형적인 인디게임의 실제 가격은 중앙값($4.99)에 가깝다. 인디 시장에서 $10을 넘는 가격 책정은 소수 선택임을 확인할 수 있다.

## 5. 그룹별 가격대 구성 비율 비교 — 누적 바 차트

In [6]:
GROUP_COLORS = {
    '$1~$5':   '#4C72B0',
    '$5~$10':  '#55A868',
    '$10~$15': '#DD8452',
    '$15~$20': '#C44E52',
    '$20~$30': '#8172B2',
    '$30~$60': '#937860',
    '$60+':    '#8C8C8C',
}

group_price = (
    df.groupby(['group', 'price_range'], observed=True)
    .agg(game_count=('appid', 'nunique'))
    .reset_index()
)
group_total = group_price.groupby('group')['game_count'].transform('sum')
group_price['ratio'] = group_price['game_count'] / group_total * 100

fig = go.Figure()

for label in PRICE_LABELS:
    subset = group_price[group_price['price_range'] == label]
    fig.add_trace(go.Bar(
        name=label,
        x=subset['group'],
        y=subset['ratio'].round(1),
        marker_color=GROUP_COLORS[label],
        text=subset['ratio'].apply(lambda v: f'{v:.1f}%'),
        textposition='inside',
        insidetextanchor='middle',
        hovertemplate=f'<b>{label}</b><br>그룹: %{{x}}<br>비율: %{{y:.1f}}%<extra></extra>',
    ))

fig.update_layout(
    barmode='stack',
    title='그룹별 가격대 구성 비율 — 초기 반응 vs 침묵 그룹<br>'
          '<sub>각 그룹 내 가격대 비율 비교</sub>',
    xaxis_title='그룹',
    yaxis_title='비율 (%)',
    height=460,
    legend_title='가격대',
)

fig.show()

**해석:** 초기 반응 그룹과 침묵 그룹의 가격대 구성 비율을 비교한다. 두 그룹 모두 ~$5 구간이 지배적이지만, 비율 차이가 있다면 특정 가격대가 유저 반응 확보에 유리하거나 불리한 조건임을 시사한다.

## 6. 첫 반응군 가격대 분포

**분석 목적:** 첫 반응(리뷰 10~49개)을 확보한 게임들의 가격대 분포를 파악하고, 침묵 그룹과 비교해 가격 포지셔닝 차이를 확인한다.

**사용 데이터:** `steam_indie_games.csv` 중 `total_reviews` 10~49개 필터 (첫 반응군 4,840개)

**시각화 방법:**
- 바: 첫 반응군 가격대별 비율 (%)
- 기준선(|): 침묵 그룹 가격대별 비율 — 바가 기준선보다 높으면 첫 반응군에서 해당 가격대가 더 많음

In [7]:
# 첫 반응군 필터링
first = df_response[
    (df_response['total_reviews'] >= 10) & (df_response['total_reviews'] <= 49)
].copy()

df_silence_local = df_silence.copy()

for subset in [first, df_silence_local]:
    subset['price'] = pd.to_numeric(subset['price'], errors='coerce')

first           = first.dropna(subset=['price'])
df_silence_local = df_silence_local.dropna(subset=['price'])

first['price_range']            = pd.cut(first['price'],            bins=PRICE_BINS, labels=PRICE_LABELS, right=False)
df_silence_local['price_range'] = pd.cut(df_silence_local['price'], bins=PRICE_BINS, labels=PRICE_LABELS, right=False)

total_first   = len(first)
total_silence = len(df_silence_local)

first_dist   = (first.groupby('price_range', observed=True)['appid'].count() / total_first   * 100).rename('first_rate')
silence_dist = (df_silence_local.groupby('price_range', observed=True)['appid'].count() / total_silence * 100).rename('silence_rate')

plot_df = pd.DataFrame({'first_rate': first_dist, 'silence_rate': silence_dist}).reset_index()
plot_df['diff'] = (plot_df['first_rate'] - plot_df['silence_rate']).round(1)

fig = go.Figure()

# 첫 반응군 바
fig.add_trace(go.Bar(
    x=plot_df['price_range'].astype(str),
    y=plot_df['first_rate'].round(1),
    name='첫 반응군 비율',
    marker_color='#4C72B0',
    text=plot_df['first_rate'].apply(lambda v: f'{v:.1f}%'),
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>첫 반응군: %{y:.1f}%<extra></extra>',
))

# 침묵 그룹 기준선 마커
fig.add_trace(go.Scatter(
    x=plot_df['price_range'].astype(str),
    y=plot_df['silence_rate'].round(1),
    mode='markers',
    name='침묵 그룹 비율',
    marker=dict(symbol='line-ns', size=14, color='#C44E52', line=dict(width=2.5, color='#C44E52')),
    hovertemplate='<b>%{x}</b><br>침묵 그룹: %{y:.1f}%<extra></extra>',
))

fig.update_layout(
    title=(
        f'첫 반응군 가격대 분포 vs 침묵 그룹 기준선<br>'
        f'<sub>첫 반응군 {total_first:,}개 (리뷰 10~49개) / 침묵 그룹 {total_silence:,}개 / |: 침묵 그룹 비율 기준선</sub>'
    ),
    xaxis_title='가격대',
    yaxis_title='게임 비율 (%)',
    height=460,
    plot_bgcolor='#FAFAFA',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()

print(f'첫 반응군 중앙값 가격: ${first["price"].median():.2f}')
print(f'침묵 그룹 중앙값 가격: ${df_silence_local["price"].median():.2f}')
print()
display(
    plot_df.rename(columns={
        'price_range': '가격대',
        'first_rate': '첫반응군(%)',
        'silence_rate': '침묵그룹(%)',
        'diff': '차이(%p, 첫반응-침묵)',
    })
    .set_index('가격대')
    .style.format({'첫반응군(%)': '{:.1f}', '침묵그룹(%)': '{:.1f}', '차이(%p, 첫반응-침묵)': '{:+.1f}'})
)

첫 반응군 중앙값 가격: $4.99
침묵 그룹 중앙값 가격: $3.99



,첫반응군(%),침묵그룹(%),"차이(%p, 첫반응-침묵)"
가격대,,,
$1~$5,56.1,70.0,-13.9
$5~$10,28.1,21.3,+6.8
$10~$15,9.8,5.2,+4.6
$15~$20,3.9,2.0,+1.9
$20~$30,1.3,0.8,+0.5
$30~$60,0.3,0.4,-0.1
$60+,0.4,0.2,+0.2


**해석:** 첫 반응군은 침묵 그룹 대비 $1~$5 비율이 낮고(56.1% vs 70.0%, −13.9%p), $5~$10 비율이 높다(28.1% vs 21.3%, +6.8%p). 침묵 그룹이 저가($1~$5)에 극단적으로 집중된 반면, 첫 반응군은 $5~$10 구간 비율이 상대적으로 더 높다. 중앙값은 첫 반응군 $4.99 / 침묵 그룹 $3.99로 $1 차이가 난다.

**주의:** 가격과 첫 반응 확보의 관계는 상관관계이며 인과관계가 아니다. $5~$10 가격대가 유리한 것이 아니라, 해당 가격대에 해당하는 장르·퀄리티 조건의 게임들이 반응을 더 많이 얻었을 가능성이 높다. (장르×가격 매트릭스)와 함께 해석해야 한다.